# 自注意力与Transformer

本notebook介绍自注意力机制和完整的Transformer架构。

## 学习目标

- 理解自注意力与交叉注意力的区别
- 掌握位置编码的原理与实现
- 对比CNN、RNN和自注意力的特性
- 实现完整的Transformer编码器-解码器

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import math

torch.manual_seed(42)

## 1. 自注意力机制

### 1.1 什么是自注意力?

**交叉注意力**(Cross-Attention): 查询、键、值来自不同来源
- 例: Seq2seq中,查询来自解码器,键值来自编码器

**自注意力**(Self-Attention): 查询、键、值来自同一序列!
- 序列 $\mathbf{x}_1, \ldots, \mathbf{x}_n$ 同时作为Q, K, V
- 每个位置 $i$ 都关注整个序列(包括自己)

### 1.2 自注意力公式

给定输入序列 $\mathbf{X} = [\mathbf{x}_1, \ldots, \mathbf{x}_n]^\top \in \mathbb{R}^{n \times d}$:

$$
\mathbf{y}_i = f(\mathbf{x}_i, (\mathbf{x}_1, \mathbf{x}_1), \ldots, (\mathbf{x}_n, \mathbf{x}_n))
$$

**矩阵形式**(使用缩放点积注意力):

$$
\text{SelfAttention}(\mathbf{X}) = \text{softmax}\left(\frac{\mathbf{X}\mathbf{X}^\top}{\sqrt{d}}\right) \mathbf{X}
$$

更一般地,使用线性投影:

$$
\begin{align}
\mathbf{Q} &= \mathbf{X}\mathbf{W}^Q \\
\mathbf{K} &= \mathbf{X}\mathbf{W}^K \\
\mathbf{V} &= \mathbf{X}\mathbf{W}^V \\
\text{SelfAttention}(\mathbf{X}) &= \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d}}\right) \mathbf{V}
\end{align}
$$

### 1.3 自注意力的优势

1. **全局感受野**: 每个位置直接关注所有其他位置
2. **并行计算**: 无需像RNN那样顺序处理
3. **最短路径**: 任意两个位置的最大路径长度为 $O(1)$
4. **灵活建模**: 动态权重适应不同输入

In [ ]:
# 简单自注意力实现
class SimpleSelfAttention(nn.Module):
    """简化的自注意力(不带多头)"""
    def __init__(self, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        
    def forward(self, X):
        """
        X: (batch_size, seq_len, embed_dim)
        返回: (batch_size, seq_len, embed_dim)
        """
        Q = self.W_q(X)  # (batch_size, seq_len, embed_dim)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 计算注意力分数
        scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(self.embed_dim)
        attention_weights = F.softmax(scores, dim=-1)
        
        # 加权求和
        output = torch.bmm(attention_weights, V)
        
        return output, attention_weights

# 测试
batch_size, seq_len, embed_dim = 2, 5, 8
X = torch.randn(batch_size, seq_len, embed_dim)

self_attn = SimpleSelfAttention(embed_dim)
output, attn_weights = self_attn(X)

print(f"输入形状: {X.shape}")
print(f"输出形状: {output.shape}")
print(f"注意力权重形状: {attn_weights.shape}")  # (2, 5, 5)

# 可视化第一个样本的注意力权重
plt.figure(figsize=(6, 5))
sns.heatmap(attn_weights[0].detach().numpy(), annot=True, fmt='.2f', 
            cmap='YlOrRd', cbar=True, square=True)
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('自注意力权重矩阵')
plt.show()

print("\n注意: 每一行是一个查询关注所有键的权重分布!")

## 2. CNN vs RNN vs Self-Attention

对比三种序列建模架构的特性:

### 2.1 特性对比

| 架构 | 计算复杂度 | 顺序操作 | 最大路径长度 | 并行性 |
|------|------------|----------|--------------|--------|
| **CNN** (核大小$k$) | $O(knd^2)$ | $O(1)$ | $O(n/k)$ | ✅ 高 |
| **RNN** | $O(nd^2)$ | $O(n)$ | $O(n)$ | ❌ 低 |
| **Self-Attention** | $O(n^2d)$ | $O(1)$ | $O(1)$ | ✅ 高 |

其中:
- $n$: 序列长度
- $d$: 特征维度
- $k$: 卷积核大小

### 2.2 详细分析

**卷积神经网络**:
- ✅ 并行计算,训练快
- ✅ 局部感受野,参数共享
- ❌ 需要堆叠多层才能捕获长距离依赖
- ❌ 最大路径长度 $O(n/k)$,对长序列不友好

**循环神经网络**:
- ✅ 天然建模序列顺序
- ✅ 理论上可处理任意长度序列
- ❌ 顺序计算,无法并行
- ❌ 梯度消失/爆炸,长距离依赖困难
- ❌ 最大路径长度 $O(n)$

**自注意力**:
- ✅ 最短路径 $O(1)$,直接建模任意依赖
- ✅ 完全并行,训练高效
- ✅ 动态权重,灵活适应输入
- ❌ 计算复杂度 $O(n^2)$,长序列计算慢
- ❌ 内存消耗大(需要存储 $n \times n$ 的注意力矩阵)
- ❌ 缺少位置信息(需要位置编码)

### 2.3 适用场景

- **CNN**: 图像、短文本、局部特征重要的任务
- **RNN**: 长序列生成、时间序列、在线处理
- **Self-Attention**: 机器翻译、文本理解、中等长度序列 (Transformer)

## 3. 位置编码

### 3.1 为什么需要位置编码?

**问题**: 自注意力是置换不变的(permutation invariant)!

- 打乱序列顺序,注意力输出不变
- 例: "猫吃鱼" 和 "鱼吃猫" 的表示相同!

**解决方案**: 在输入中注入位置信息

$$
\mathbf{X}' = \mathbf{X} + \mathbf{P}
$$

其中 $\mathbf{P} \in \mathbb{R}^{n \times d}$ 是位置编码矩阵。

### 3.2 正弦位置编码(Sinusoidal Positional Encoding)

Transformer使用的固定位置编码:

$$
\begin{align}
\mathbf{P}_{i, 2j} &= \sin\left(\frac{i}{10000^{2j/d}}\right) \\
\mathbf{P}_{i, 2j+1} &= \cos\left(\frac{i}{10000^{2j/d}}\right)
\end{align}
$$

其中:
- $i$: 位置索引 (0, 1, ..., n-1)
- $j$: 维度索引 (0, 1, ..., d/2-1)
- 偶数维用sin,奇数维用cos

### 3.3 设计动机

1. **频率递减**: 不同维度使用不同频率
   - 低维(高频): 捕获局部位置信息
   - 高维(低频): 捕获全局位置信息

2. **相对位置**: 任意位置偏移 $\delta$ 可通过线性变换表示
   $$
   \begin{bmatrix} \cos(\delta\omega) & \sin(\delta\omega) \\ -\sin(\delta\omega) & \cos(\delta\omega) \end{bmatrix}
   \begin{bmatrix} \mathbf{P}_{i,2j} \\ \mathbf{P}_{i,2j+1} \end{bmatrix}
   = \begin{bmatrix} \mathbf{P}_{i+\delta,2j} \\ \mathbf{P}_{i+\delta,2j+1} \end{bmatrix}
   $$

3. **外推性**: 可处理训练时未见过的序列长度

4. **无需学习**: 减少参数量,避免过拟合

In [ ]:
class PositionalEncoding(nn.Module):
    """正弦位置编码"""
    def __init__(self, num_hiddens, dropout=0.1, max_len=1000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # 创建位置编码矩阵
        self.P = torch.zeros((1, max_len, num_hiddens))
        
        # 位置索引: (max_len, 1)
        position = torch.arange(max_len, dtype=torch.float32).reshape(-1, 1)
        
        # 分母项: 10000^(2j/d)
        div_term = torch.pow(10000, torch.arange(0, num_hiddens, 2, dtype=torch.float32) / num_hiddens)
        
        # 偶数维用sin
        self.P[:, :, 0::2] = torch.sin(position / div_term)
        # 奇数维用cos
        self.P[:, :, 1::2] = torch.cos(position / div_term)
        
    def forward(self, X):
        """
        X: (batch_size, seq_len, num_hiddens)
        """
        X = X + self.P[:, :X.shape[1], :].to(X.device)
        return self.dropout(X)

# 可视化位置编码
num_hiddens, max_len = 32, 60
pos_encoding = PositionalEncoding(num_hiddens, dropout=0)
pos_encoding.eval()

X = torch.zeros((1, max_len, num_hiddens))
X = pos_encoding(X)
P = pos_encoding.P[0, :max_len, :].numpy()

# 绘制不同维度的位置编码
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图: 选择几个维度的曲线
ax1 = axes[0]
selected_dims = [6, 7, 8, 9]
for dim in selected_dims:
    ax1.plot(P[:, dim], label=f'Dim {dim}')
ax1.set_xlabel('Position')
ax1.set_ylabel('Encoding Value')
ax1.set_title('不同维度的位置编码曲线')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 右图: 热力图
ax2 = axes[1]
im = ax2.imshow(P.T, aspect='auto', cmap='RdBu', interpolation='nearest')
ax2.set_xlabel('Position')
ax2.set_ylabel('Encoding Dimension')
ax2.set_title('位置编码热力图')
plt.colorbar(im, ax=ax2)

plt.tight_layout()
plt.show()

print("观察:")
print("- 低维(上方): 高频振荡,捕获细粒度位置")
print("- 高维(下方): 低频变化,捕获粗粒度位置")
print("- 类似二进制编码,但连续可微!")

In [ ]:
# 验证相对位置性质
def relative_position_linear_transform(P, i, delta, j):
    """验证位置i和i+delta在维度2j,2j+1上的线性关系"""
    omega = 1.0 / (10000 ** (2 * j / num_hiddens))
    
    # 旋转矩阵
    rotation_matrix = np.array([
        [np.cos(delta * omega), np.sin(delta * omega)],
        [-np.sin(delta * omega), np.cos(delta * omega)]
    ])
    
    # P[i, 2j:2j+2]
    p_i = P[i, 2*j:2*j+2]
    
    # P[i+delta, 2j:2j+2]
    p_i_delta = P[i + delta, 2*j:2*j+2]
    
    # 通过旋转得到
    p_i_delta_pred = rotation_matrix @ p_i
    
    error = np.linalg.norm(p_i_delta - p_i_delta_pred)
    return error

# 测试
i, delta, j = 10, 5, 3
error = relative_position_linear_transform(P, i, delta, j)
print(f"位置{i}和位置{i+delta}在维度{2*j},{2*j+1}上的线性变换误差: {error:.6f}")
print("误差应接近0,验证了相对位置编码的线性性质!")

## 4. Transformer架构

### 4.1 整体架构

Transformer = 编码器 + 解码器

**编码器**:
- $N$ 个相同的层堆叠
- 每层包含:
  1. 多头自注意力
  2. 位置前馈网络(FFN)
  3. 残差连接 + 层归一化

**解码器**:
- $N$ 个相同的层堆叠
- 每层包含:
  1. 掩码多头自注意力(防止看到未来信息)
  2. 多头交叉注意力(编码器-解码器注意力)
  3. 位置前馈网络
  4. 残差连接 + 层归一化

### 4.2 关键组件

**1. 位置前馈网络(Position-wise FFN)**:
$$
\text{FFN}(\mathbf{x}) = \max(0, \mathbf{x}\mathbf{W}_1 + \mathbf{b}_1)\mathbf{W}_2 + \mathbf{b}_2
$$

- 对每个位置独立应用
- 两层全连接,中间用ReLU
- 通常中间维度是4倍模型维度

**2. 残差连接(Residual Connection)**:
$$
\mathbf{y} = \text{LayerNorm}(\mathbf{x} + \text{Sublayer}(\mathbf{x}))
$$

- 缓解梯度消失
- 允许堆叠更深的网络

**3. 层归一化(Layer Normalization)**:
$$
\text{LayerNorm}(\mathbf{x}) = \frac{\mathbf{x} - \mu}{\sqrt{\sigma^2 + \epsilon}} \odot \gamma + \beta
$$

- 在特征维度归一化(vs BatchNorm在批次维度)
- 对变长序列更友好

**4. 掩码自注意力(Masked Self-Attention)**:
- 解码器训练时使用
- 防止位置 $i$ 看到位置 $>i$ 的信息
- 保持自回归性质

In [ ]:
# 复用之前的多头注意力
class MultiHeadAttention(nn.Module):
    def __init__(self, num_hiddens, num_heads, dropout=0.1):
        super().__init__()
        assert num_hiddens % num_heads == 0
        self.num_heads = num_heads
        self.d_k = num_hiddens // num_heads
        
        self.W_q = nn.Linear(num_hiddens, num_hiddens)
        self.W_k = nn.Linear(num_hiddens, num_hiddens)
        self.W_v = nn.Linear(num_hiddens, num_hiddens)
        self.W_o = nn.Linear(num_hiddens, num_hiddens)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, queries, keys, values, mask=None):
        batch_size = queries.size(0)
        
        # 线性投影并分头: (batch, seq_len, num_heads, d_k)
        Q = self.W_q(queries).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(keys).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(values).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 缩放点积注意力: (batch, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # 应用掩码
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # 加权求和: (batch, num_heads, seq_len, d_k)
        output = torch.matmul(attn_weights, V)
        
        # 合并多头: (batch, seq_len, num_hiddens)
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)
        
        return self.W_o(output)

class PositionWiseFFN(nn.Module):
    """位置前馈网络"""
    def __init__(self, num_hiddens, ffn_num_hiddens, dropout=0.1):
        super().__init__()
        self.dense1 = nn.Linear(num_hiddens, ffn_num_hiddens)
        self.relu = nn.ReLU()
        self.dense2 = nn.Linear(ffn_num_hiddens, num_hiddens)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, X):
        return self.dense2(self.dropout(self.relu(self.dense1(X))))

class AddNorm(nn.Module):
    """残差连接 + 层归一化"""
    def __init__(self, num_hiddens, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(num_hiddens)
        
    def forward(self, X, Y):
        return self.ln(self.dropout(Y) + X)

In [ ]:
class EncoderBlock(nn.Module):
    """Transformer编码器块"""
    def __init__(self, num_hiddens, num_heads, ffn_num_hiddens, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(num_hiddens, num_heads, dropout)
        self.addnorm1 = AddNorm(num_hiddens, dropout)
        self.ffn = PositionWiseFFN(num_hiddens, ffn_num_hiddens, dropout)
        self.addnorm2 = AddNorm(num_hiddens, dropout)
        
    def forward(self, X, mask=None):
        # 多头自注意力
        Y = self.addnorm1(X, self.attention(X, X, X, mask))
        # 位置前馈网络
        return self.addnorm2(Y, self.ffn(Y))

class TransformerEncoder(nn.Module):
    """Transformer编码器"""
    def __init__(self, vocab_size, num_hiddens, num_heads, num_layers, 
                 ffn_num_hiddens, dropout=0.1, max_len=1000):
        super().__init__()
        self.num_hiddens = num_hiddens
        self.embedding = nn.Embedding(vocab_size, num_hiddens)
        self.pos_encoding = PositionalEncoding(num_hiddens, dropout, max_len)
        
        self.blocks = nn.ModuleList([
            EncoderBlock(num_hiddens, num_heads, ffn_num_hiddens, dropout)
            for _ in range(num_layers)
        ])
        
    def forward(self, X, mask=None):
        # 嵌入 + 位置编码(缩放嵌入)
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.num_hiddens))
        
        # 通过所有编码器块
        for block in self.blocks:
            X = block(X, mask)
        
        return X

# 测试编码器
vocab_size, num_hiddens, num_heads = 200, 64, 8
num_layers, ffn_num_hiddens = 2, 256
batch_size, seq_len = 2, 10

encoder = TransformerEncoder(vocab_size, num_hiddens, num_heads, num_layers, ffn_num_hiddens)
X = torch.randint(0, vocab_size, (batch_size, seq_len))
output = encoder(X)

print(f"编码器输入形状: {X.shape}")
print(f"编码器输出形状: {output.shape}")  # (2, 10, 64)
print(f"\n参数量: {sum(p.numel() for p in encoder.parameters()):,}")

In [ ]:
class DecoderBlock(nn.Module):
    """Transformer解码器块"""
    def __init__(self, num_hiddens, num_heads, ffn_num_hiddens, dropout=0.1):
        super().__init__()
        # 掩码自注意力
        self.attention1 = MultiHeadAttention(num_hiddens, num_heads, dropout)
        self.addnorm1 = AddNorm(num_hiddens, dropout)
        # 编码器-解码器注意力
        self.attention2 = MultiHeadAttention(num_hiddens, num_heads, dropout)
        self.addnorm2 = AddNorm(num_hiddens, dropout)
        # 前馈网络
        self.ffn = PositionWiseFFN(num_hiddens, ffn_num_hiddens, dropout)
        self.addnorm3 = AddNorm(num_hiddens, dropout)
        
    def forward(self, X, enc_output, src_mask=None, tgt_mask=None):
        # 掩码自注意力
        Y = self.addnorm1(X, self.attention1(X, X, X, tgt_mask))
        # 编码器-解码器注意力
        Y = self.addnorm2(Y, self.attention2(Y, enc_output, enc_output, src_mask))
        # 前馈网络
        return self.addnorm3(Y, self.ffn(Y))

class TransformerDecoder(nn.Module):
    """Transformer解码器"""
    def __init__(self, vocab_size, num_hiddens, num_heads, num_layers,
                 ffn_num_hiddens, dropout=0.1, max_len=1000):
        super().__init__()
        self.num_hiddens = num_hiddens
        self.embedding = nn.Embedding(vocab_size, num_hiddens)
        self.pos_encoding = PositionalEncoding(num_hiddens, dropout, max_len)
        
        self.blocks = nn.ModuleList([
            DecoderBlock(num_hiddens, num_heads, ffn_num_hiddens, dropout)
            for _ in range(num_layers)
        ])
        
        self.dense = nn.Linear(num_hiddens, vocab_size)
        
    def forward(self, X, enc_output, src_mask=None, tgt_mask=None):
        # 嵌入 + 位置编码
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.num_hiddens))
        
        # 通过所有解码器块
        for block in self.blocks:
            X = block(X, enc_output, src_mask, tgt_mask)
        
        # 输出层
        return self.dense(X)

# 测试解码器
decoder = TransformerDecoder(vocab_size, num_hiddens, num_heads, num_layers, ffn_num_hiddens)
Y = torch.randint(0, vocab_size, (batch_size, seq_len))

# 创建目标掩码(下三角矩阵)
tgt_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)

dec_output = decoder(Y, output, tgt_mask=tgt_mask)

print(f"解码器输入形状: {Y.shape}")
print(f"解码器输出形状: {dec_output.shape}")  # (2, 10, 200)
print(f"\n参数量: {sum(p.numel() for p in decoder.parameters()):,}")

## 5. 完整Transformer模型

将编码器和解码器组合成完整的Transformer。

In [ ]:
class Transformer(nn.Module):
    """完整的Transformer模型"""
    def __init__(self, src_vocab_size, tgt_vocab_size, num_hiddens, num_heads,
                 num_layers, ffn_num_hiddens, dropout=0.1, max_len=1000):
        super().__init__()
        self.encoder = TransformerEncoder(
            src_vocab_size, num_hiddens, num_heads, num_layers, 
            ffn_num_hiddens, dropout, max_len
        )
        self.decoder = TransformerDecoder(
            tgt_vocab_size, num_hiddens, num_heads, num_layers,
            ffn_num_hiddens, dropout, max_len
        )
        
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        """
        src: (batch_size, src_len)
        tgt: (batch_size, tgt_len)
        """
        enc_output = self.encoder(src, src_mask)
        dec_output = self.decoder(tgt, enc_output, src_mask, tgt_mask)
        return dec_output
    
    def generate_square_subsequent_mask(self, sz):
        """生成目标掩码(下三角)"""
        mask = torch.tril(torch.ones(sz, sz))
        return mask.unsqueeze(0).unsqueeze(0)

# 创建完整模型
src_vocab_size, tgt_vocab_size = 5000, 5000
num_hiddens, num_heads = 512, 8
num_layers, ffn_num_hiddens = 6, 2048

model = Transformer(
    src_vocab_size, tgt_vocab_size, num_hiddens, num_heads,
    num_layers, ffn_num_hiddens, dropout=0.1
)

# 测试前向传播
batch_size, src_len, tgt_len = 32, 20, 15
src = torch.randint(0, src_vocab_size, (batch_size, src_len))
tgt = torch.randint(0, tgt_vocab_size, (batch_size, tgt_len))
tgt_mask = model.generate_square_subsequent_mask(tgt_len)

output = model(src, tgt, tgt_mask=tgt_mask)

print(f"源序列形状: {src.shape}")
print(f"目标序列形状: {tgt.shape}")
print(f"模型输出形状: {output.shape}")  # (32, 15, 5000)
print(f"\n总参数量: {sum(p.numel() for p in model.parameters()):,}")
print("\n这是一个标准的Transformer Base模型!")

## 6. 可视化掩码机制

理解解码器的掩码自注意力如何防止"偷看"未来信息。

In [ ]:
# 可视化不同类型的掩码
seq_len = 8

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. 无掩码(编码器自注意力)
no_mask = torch.ones(seq_len, seq_len)
axes[0].imshow(no_mask, cmap='Greens', vmin=0, vmax=1)
axes[0].set_title('编码器自注意力\n(无掩码,全连接)')
axes[0].set_xlabel('Key Position')
axes[0].set_ylabel('Query Position')

# 2. 下三角掩码(解码器自注意力)
causal_mask = torch.tril(torch.ones(seq_len, seq_len))
axes[1].imshow(causal_mask, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('解码器自注意力\n(因果掩码,防止看到未来)')
axes[1].set_xlabel('Key Position')
axes[1].set_ylabel('Query Position')

# 3. Padding掩码示例
# 假设序列[1,2,3,4,5,<pad>,<pad>,<pad>]
valid_len = 5
padding_mask = torch.zeros(seq_len, seq_len)
padding_mask[:, :valid_len] = 1
axes[2].imshow(padding_mask, cmap='Oranges', vmin=0, vmax=1)
axes[2].set_title('Padding掩码\n(忽略填充位置)')
axes[2].set_xlabel('Key Position')
axes[2].set_ylabel('Query Position')

for ax in axes:
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.grid(False)

plt.tight_layout()
plt.show()

print("\n掩码说明:")
print("- 绿色(编码器): 每个位置可以看到所有位置")
print("- 蓝色(解码器): 位置i只能看到位置≤i(自回归)")
print("- 橙色(Padding): 所有位置都忽略填充部分")

## 7. 小结

### 自注意力核心

1. **Query = Key = Value**: 序列关注自己
2. **并行计算**: $O(1)$ 顺序操作,训练高效
3. **全局感受野**: 直接建模任意距离的依赖
4. **位置不变**: 需要位置编码注入顺序信息

### 位置编码要点

1. **正弦编码**: 使用sin/cos函数,不需学习
2. **频率递减**: 不同维度捕获不同粒度的位置信息
3. **相对位置**: 支持线性变换表示相对偏移
4. **外推性**: 可处理训练时未见过的长度

### Transformer优势

1. **并行性**: 编码器和解码器都可并行
2. **长距离**: 最大路径长度 $O(1)$
3. **可解释性**: 注意力权重可视化
4. **迁移性**: 预训练模型(BERT, GPT)效果好

### Transformer局限

1. **计算复杂度**: $O(n^2)$,长序列昂贵
2. **内存消耗**: 需要存储 $n \times n$ 矩阵
3. **数据需求**: 需要大量数据才能充分训练

### 应用领域

- **NLP**: 机器翻译、文本生成、问答系统
- **CV**: Vision Transformer (ViT)、图像分类
- **语音**: 语音识别、语音合成
- **多模态**: CLIP、DALL-E

### 改进方向

- **长序列**: Linformer, Performer, Longformer (降低复杂度)
- **效率**: Flash Attention (优化内存访问)
- **稀疏注意力**: 只关注部分位置
- **相对位置**: 改进位置编码(如RoPE)

## 练习

1. **可学习位置编码**: 将固定的正弦编码替换为可学习的Embedding,比较两者性能。

2. **复杂度分析**: 实际测量不同序列长度(64, 128, 256, 512)下的计算时间和内存消耗。

3. **注意力可视化**: 在训练好的模型上,可视化不同层、不同头的注意力模式。

4. **层数实验**: 比较不同层数(2, 4, 6, 12)的Transformer在翻译任务上的性能。

5. **掩码实验**: 尝试在解码器中移除因果掩码,观察对生成质量的影响。

6. **预归一化**: 实现Pre-LN (归一化在子层之前)并与Post-LN (归一化在子层之后)比较。